# Docling

[Docling](https://github.com/DS4SD/docling) parses PDF, DOCX, PPTX, HTML, and other formats into a rich unified representation including document layout, tables etc., making them ready for generative AI workflows like RAG.

This integration provides Docling's capabilities via the `DoclingLoader` document loader.

## Overview

<!--
### Integration details

| Class | Package | Local | Serializable | JS support|
| :--- | :--- | :---: | :---: |  :---: |
| langchain_docling.DoclingLoader | langchain-docling | ✅ | ❌ | ❌ |

### Loader features
| Source | Document Lazy Loading | Native Async Support
| :---: | :---: | :---: |
| DoclingLoader | ✅ | ❌ |
 -->

The presented `DoclingLoader` component enables you to:
- use various document types in your LLM applications with ease and speed, and
- leverage Docling's rich format for advanced, document-native grounding.

`DoclingLoader` supports two different export modes:
- `ExportType.DOC_CHUNKS` (default): if you want to have each input document chunked and
  to then capture each individual chunk as a separate LangChain Document downstream, or
- `ExportType.MARKDOWN`: if you want to capture each input document as a separate
  LangChain Document

The example allows exploring both modes via parameter `EXPORT_TYPE`; depending on the
value set, the example pipeline is then set up accordingly.

## Setup

For advanced usage, `DoclingLoader` has the following parameters:
- `file_path`: source as single str (URL or local file) or iterable thereof
- `converter` (optional): any specific Docling converter instance to use
- `convert_kwargs` (optional): any specific kwargs for conversion execution
- `export_type` (optional): export mode to use: `ExportType.DOC_CHUNKS` (default) or
    `ExportType.MARKDOWN`
- `md_export_kwargs` (optional): any specific Markdown export kwargs (for Markdown mode)
- `chunker` (optional): any specific Docling chunker instance to use (for doc-chunk
    mode)
- `meta_extractor` (optional): any specific metadata extractor to use


---
#### RAG using Docling
---

In [20]:
# %% [markdown]
# # Step 1. Install Dependencies

# %%
%pip install -qU langchain langchain-openai langchain-docling langchain-milvus pymilvus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.1 MB/s eta 0:00:00


In [21]:
# %% [markdown]
# # Step 2. Imports & Environment Setup

# %%
import os
from pathlib import Path
from tempfile import mkdtemp

from dotenv import load_dotenv

from langchain_core.prompts import PromptTemplate
from langchain_docling.loader import ExportType, DoclingLoader
from docling.chunking import HybridChunker

In [22]:
# Load environment variables
load_dotenv()

False

In [24]:
# Disable tokenizers parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [25]:
# %% [markdown]
# # Step 3. Configuration

# %%
FILE_PATH      = ["https://arxiv.org/pdf/2408.09869"]  # Docling Technical Report
EXPORT_TYPE    = ExportType.DOC_CHUNKS
GEN_MODEL_ID   = "gpt-4o-mini"    # OpenAI model for LLM
EMBED_MODEL_ID = "text-embedding-3-small"  # OpenAI embeddings
QUESTION       = "Which are the main AI models in Docling?"
TOP_K          = 3
MILVUS_URI     = str(Path(mkdtemp()) / "docling.db")

PROMPT = PromptTemplate.from_template(
    "Context information is below.\n---------------------\n{context}\n"
    "---------------------\nGiven the context information and not prior knowledge, "
    "answer the query.\nQuery: {input}\nAnswer:\n"
)


In [27]:
# %% [markdown]
# # Step 4. Load Documents with Docling

# %%
loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type=EXPORT_TYPE,
    chunker=HybridChunker(max_tokens=512),  # ✅ explicitly set max_tokens
)

docs = loader.load()

# Handle export type
if EXPORT_TYPE == ExportType.DOC_CHUNKS:
    splits = docs
else:
    raise ValueError(f"Unexpected export type: {EXPORT_TYPE}")




/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Token indices sequence length is longer than the specified maximum sequence length for this model (656 > 512). Running this sequence through the model will result in indexing errors


Loaded 27 chunks
- Version 1.0
Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubha...
- Abstract
This technical report introduces Docling , an easy to use, self-contained, MITlicensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models...
- 1 Introduction
Converting PDF documents back into a machine-processable format has been a major challenge for decades due to their huge variability in formats, weak standardization and printing-optimi...


In [34]:
print(f"Loaded {len(splits)} chunks")
for d in splits[:3]:
    print(f"- {d.page_content[:200]}...")

Loaded 27 chunks
- Version 1.0
Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubha...
- Abstract
This technical report introduces Docling , an easy to use, self-contained, MITlicensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models...
- 1 Introduction
Converting PDF documents back into a machine-processable format has been a major challenge for decades due to their huge variability in formats, weak standardization and printing-optimi...


In [35]:
import pandas as pd

In [36]:
# Create a dataframe with chunk content and metadata
df = pd.DataFrame([
    {
        "chunk_id": i,
        "text": d.page_content,
        **d.metadata  # add all metadata as columns
    }
    for i, d in enumerate(splits)
])

In [37]:
# Show first 10 chunks
df.sample(10)

,chunk_id,text,source,dl_meta
7,7,Layout Analysis Model\nOur layout analysis mod...,https://arxiv.org/pdf/2408.09869,{'schema_name': 'docling_core.transforms.chunk...
4,4,3 Processing pipeline\nDocling implements a li...,https://arxiv.org/pdf/2408.09869,{'schema_name': 'docling_core.transforms.chunk...
19,19,References\n- [15] P. Team. PyPDFium2: Python ...,https://arxiv.org/pdf/2408.09869,{'schema_name': 'docling_core.transforms.chunk...
16,16,References\n- [1] J. AI. Easyocr: Ready-to-use...,https://arxiv.org/pdf/2408.09869,{'schema_name': 'docling_core.transforms.chunk...
21,21,Appendix\nTable 2: Prediction performance (mAP...,https://arxiv.org/pdf/2408.09869,{'schema_name': 'docling_core.transforms.chunk...
22,22,Appendix\nto avoid this at any cost in order t...,https://arxiv.org/pdf/2408.09869,{'schema_name': 'docling_core.transforms.chunk...
13,13,4 Performance\nTable 1: Runtime characteristic...,https://arxiv.org/pdf/2408.09869,{'schema_name': 'docling_core.transforms.chunk...
15,15,6 Future work and contributions\nDocling is de...,https://arxiv.org/pdf/2408.09869,{'schema_name': 'docling_core.transforms.chunk...
3,3,"2 Getting Started\nTo use Docling, you can sim...",https://arxiv.org/pdf/2408.09869,{'schema_name': 'docling_core.transforms.chunk...
2,2,1 Introduction\nConverting PDF documents back ...,https://arxiv.org/pdf/2408.09869,{'schema_name': 'docling_core.transforms.chunk...


In [39]:
%pip install pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.3 MB/s eta 0:00:00


In [48]:
from pyvis.network import Network
from IPython.core.display import display, HTML

# Create network
net_graph = Network(height="600px", width="100%", notebook=True)

for i, doc in enumerate(splits[:50]):  # limit first 50 chunks
    tooltip_text = doc.page_content[:200] + "..."
    net_graph.add_node(i, label=f"Chunk {i}", title=tooltip_text)

# Save to temporary HTML
tmp_file = "/tmp/doc_chunks.html"
net_graph.save_graph(tmp_file)

# Display in notebook
#display(HTML(f'<iframe src="{tmp_file}" width="100%" height="600px"></iframe>'))

In [47]:
from google.colab import files
files.download("doc_chunks.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Using Panel display**

In [49]:
from rich import print
from rich.panel import Panel

In [50]:
# Display first 10 chunks with metadata
for i, doc in enumerate(splits[:10]):
    meta_text = "\n".join(f"{k}: {v}" for k, v in doc.metadata.items() if k != "pk")
    content_preview = doc.page_content[:500]  # limit to first 500 chars
    panel_text = f"{content_preview}\n\n[bold green]Metadata:[/bold green]\n{meta_text}"

    print(Panel(panel_text, title=f"Chunk {i}", width=100))

╭──────────────────────────────────────────── Chunk 0 ─────────────────────────────────────────────╮
│ Version 1.0                                                                                      │
│ Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar   │
│ Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham   │
│ Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. │
│ J. Staar                                                                                         │
│ AI4K Group, IBM Research R¨ uschlikon, Switzerland                                               │
│                                                                                                  │
│ Metadata:                                                                                        │
│ source: https://arxiv.org/pdf/2408.09869                                                         │
│ dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0',          │
│ 'doc_items': [{'self_ref': '#/texts/3', 'parent': {'$ref': '#/body'}, 'children': [],            │
│ 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 113.643, 't':    │
│ 481.532, 'r': 498.359, 'b': 439.849, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 295]}]},     │
│ {'self_ref': '#/texts/4', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', │
│ 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 249.283, 't': 427.545, 'r': 362.717, 'b': │
│ 408.084, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 50]}]}], 'headings': ['Version 1.0'],    │
│ 'origin': {'mimetype': 'application/pdf', 'binary_hash': 11465328351749295394, 'filename':       │
│ '2408.09869v5.pdf'}}                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Chunk 1 ─────────────────────────────────────────────╮
│ Abstract                                                                                         │
│ This technical report introduces Docling , an easy to use, self-contained, MITlicensed           │
│ open-source package for PDF document conversion. It is powered by state-of-the-art specialized   │
│ AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and     │
│ runs efficiently on commodity hardware in a small resource budget. The code interface allows for │
│ easy extensibility and addition of new features and models.                                      │
│                                                                                                  │
│ Metadata:                                                                                        │
│ source: https://arxiv.org/pdf/2408.09869                                                         │
│ dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0',          │
│ 'doc_items': [{'self_ref': '#/texts/6', 'parent': {'$ref': '#/body'}, 'children': [],            │
│ 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 143.865, 't':    │
│ 364.013, 'r': 468.138, 'b': 300.737, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 431]}]}],    │
│ 'headings': ['Abstract'], 'origin': {'mimetype': 'application/pdf', 'binary_hash':               │
│ 11465328351749295394, 'filename': '2408.09869v5.pdf'}}                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Chunk 2 ─────────────────────────────────────────────╮
│ 1 Introduction                                                                                   │
│ Converting PDF documents back into a machine-processable format has been a major challenge for   │
│ decades due to their huge variability in formats, weak standardization and printing-optimized    │
│ characteristic, which discards most structural features and metadata. With the advent of LLMs    │
│ and popular application patterns such as retrieval-augmented generation (RAG), leveraging the    │
│ rich content embedded in PDFs has become ever more relevant. In the past decade, several         │
│ powerful document u                                                                              │
│                                                                                                  │
│ Metadata:                                                                                        │
│ source: https://arxiv.org/pdf/2408.09869                                                         │
│ dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0',          │
│ 'doc_items': [{'self_ref': '#/texts/8', 'parent': {'$ref': '#/body'}, 'children': [],            │
│ 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 108.0, 't':      │
│ 239.37, 'r': 504.003, 'b': 143.54600000000005, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0,    │
│ 792]}]}, {'self_ref': '#/texts/9', 'parent': {'$ref': '#/body'}, 'children': [],                 │
│ 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 108.0, 't':      │
│ 135.88800000000003, 'r': 504.003, 'b': 83.52099999999996, 'coord_origin': 'BOTTOMLEFT'},         │
│ 'charspan': [0, 488]}]}, {'self_ref': '#/texts/12', 'parent': {'$ref': '#/body'}, 'children':    │
│ [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 2, 'bbox': {'l': 108.0, 't':  │
│ 716.523, 'r': 253.972, 'b': 707.971, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 36]}]},      │
│ {'self_ref': '#/texts/13', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer':    │
│ 'body', 'label': 'list_item', 'prov': [{'page_no': 2, 'bbox': {'l': 135.397, 't': 695.23, 'r':   │
│ 468.397, 'b': 686.678, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 78]}]}, {'self_ref':       │
│ '#/texts/14', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body',         │
│ 'label': 'list_item', 'prov': [{'page_no': 2, 'bbox': {'l': 135.397, 't': 680.366, 'r': 504.003, │
│ 'b': 660.905, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 96]}]}, {'self_ref': '#/texts/15',  │
│ 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', │
│ 'prov': [{'page_no': 2, 'bbox': {'l': 135.397, 't': 654.593, 'r': 480.85, 'b': 646.041,          │
│ 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 86]}]}, {'self_ref': '#/texts/16', 'parent':      │
│ {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov':   │
│ [{'page_no': 2, 'bbox': {'l': 135.397, 't': 639.729, 'r': 333.463, 'b': 631.177, 'coord_origin': │
│ 'BOTTOMLEFT'}, 'charspan': [0, 47]}]}, {'self_ref': '#/texts/17', 'parent': {'$ref':             │
│ '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov':            │
│ [{'page_no': 2, 'bbox': {'l': 135.397, 't': 624.866, 'r': 504.003, 'b': 605.405, 'coord_origin': │
│ 'BOTTOMLEFT'}, 'charspan': [0, 161]}]}, {'self_ref': '#/texts/18', 'parent': {'$ref':            │
│ '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov':            │
│ [{'page_no': 2, 'bbox': {'l': 135.397, 't': 599.093, 'r': 355.411, 'b': 590.541, 'coord_origin': │
│ 'BOTTOMLEFT'}, 'charspan': [0, 54]}]}], 'headings': ['1 Introduction'], 'origin': {'mimetype':   │
│ 'application/pdf', 'binary_hash': 11465328351749295394, 'fi

╭──────────────────────────────────────────── Chunk 3 ─────────────────────────────────────────────╮
│ 2 Getting Started                                                                                │
│ To use Docling, you can simply install the docling package from PyPI. Documentation and examples │
│ are available in our GitHub repository at github.com/DS4SD/docling. All required model assets 1  │
│ are downloaded to a local huggingface datasets cache on first use, unless you choose to          │
│ pre-install the model assets in advance.                                                         │
│ Docling provides an easy code interface to convert PDF documents from file system, URLs or       │
│ binary streams, and retrieve the output in either JSON or Markdown fo                            │
│                                                                                                  │
│ Metadata:                                                                                        │
│ source: https://arxiv.org/pdf/2408.09869                                                         │
│ dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0',          │
│ 'doc_items': [{'self_ref': '#/texts/20', 'parent': {'$ref': '#/body'}, 'children': [],           │
│ 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 2, 'bbox': {'l': 108.0, 't':      │
│ 547.82, 'r': 504.003, 'b': 506.362, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 321]}]},      │
│ {'self_ref': '#/texts/21', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer':        │
│ 'body', 'label': 'text', 'prov': [{'page_no': 2, 'bbox': {'l': 108.0, 't': 498.525, 'r':         │
│ 504.003, 'b': 457.246, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 371]}]}, {'self_ref':      │
│ '#/texts/22', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label':    │
│ 'text', 'prov': [{'page_no': 2, 'bbox': {'l': 108.753, 't': 448.911, 'r': 423.447, 'b': 441.442, │
│ 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 56]}]}, {'self_ref': '#/texts/23', 'parent':      │
│ {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'code', 'prov':            │
│ [{'page_no': 2, 'bbox': {'l': 108.785, 't': 428.985, 'r': 491.336, 'b': 381.666, 'coord_origin': │
│ 'BOTTOMLEFT'}, 'charspan': [0, 265]}]}, {'self_ref': '#/texts/24', 'parent': {'$ref': '#/body'}, │
│ 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 2, 'bbox': {'l':  │
│ 108.0, 't': 367.837, 'r': 504.003, 'b': 315.649, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0,  │
│ 403]}]}], 'headings': ['2 Getting Started'], 'origin': {'mimetype': 'application/pdf',           │
│ 'binary_hash': 11465328351749295394, 'filename': '2408.09869v5.pdf'}}                            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Chunk 4 ─────────────────────────────────────────────╮
│ 3 Processing pipeline                                                                            │
│ Docling implements a linear pipeline of operations, which execute sequentially on each given     │
│ document (see Fig. 1). Each document is first parsed by a PDF backend, which retrieves the       │
│ programmatic text tokens, consisting of string content and its coordinates on the page, and also │
│ renders a bitmap image of each page to support downstream operations. Then, the standard model   │
│ pipeline applies a sequence of AI models independently on every page in the document to extract  │
│ featur                                                                                           │
│                                                                                                  │
│ Metadata:                                                                                        │
│ source: https://arxiv.org/pdf/2408.09869                                                         │
│ dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0',          │
│ 'doc_items': [{'self_ref': '#/texts/26', 'parent': {'$ref': '#/body'}, 'children': [],           │
│ 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 2, 'bbox': {'l': 108.0, 't':      │
│ 272.749, 'r': 504.003, 'b': 176.92399999999998, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0,   │
│ 796]}]}], 'headings': ['3 Processing pipeline'], 'origin': {'mimetype': 'application/pdf',       │
│ 'binary_hash': 11465328351749295394, 'filename': '2408.09869v5.pdf'}}                            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Chunk 5 ─────────────────────────────────────────────╮
│ 3.1 PDF backends                                                                                 │
│ Two basic requirements to process PDF documents in our pipeline are a) to retrieve all text      │
│ content and their geometric coordinates on each page and b) to render the visual representation  │
│ of each page as it would appear in a PDF viewer. Both these requirements are encapsulated in     │
│ Docling's PDF backend interface. While there are several open-source PDF parsing libraries       │
│ available for python, we faced major obstacles with all of them for different reasons, among     │
│ which were restric                                                                               │
│                                                                                                  │
│ Metadata:                                                                                        │
│ source: https://arxiv.org/pdf/2408.09869                                                         │
│ dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0',          │
│ 'doc_items': [{'self_ref': '#/texts/28', 'parent': {'$ref': '#/body'}, 'children': [],           │
│ 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 2, 'bbox': {'l': 108.0, 't':      │
│ 141.07100000000003, 'r': 504.003, 'b': 88.88200000000006, 'coord_origin': 'BOTTOMLEFT'},         │
│ 'charspan': [0, 487]}]}, {'self_ref': '#/texts/29', 'parent': {'$ref': '#/body'}, 'children':    │
│ [], 'content_layer': 'body', 'label': 'footnote', 'prov': [{'page_no': 2, 'bbox': {'l': 120.653, │
│ 't': 79.70000000000005, 'r': 276.461, 'b': 70.13999999999999, 'coord_origin': 'BOTTOMLEFT'},     │
│ 'charspan': [0, 42]}]}, {'self_ref': '#/texts/31', 'parent': {'$ref': '#/pictures/1'},           │
│ 'children': [], 'content_layer': 'body', 'label': 'caption', 'prov': [{'page_no': 3, 'bbox':     │
│ {'l': 108.0, 't': 570.003, 'r': 504.003, 'b': 550.542, 'coord_origin': 'BOTTOMLEFT'},            │
│ 'charspan': [0, 134]}]}, {'self_ref': '#/texts/47', 'parent': {'$ref': '#/body'}, 'children':    │
│ [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 3, 'bbox': {'l': 108.0, 't':  │
│ 524.405, 'r': 504.003, 'b': 504.943, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 173]}]},     │
│ {'self_ref': '#/texts/48', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer':        │
│ 'body', 'label': 'text', 'prov': [{'page_no': 3, 'bbox': {'l': 108.0, 't': 497.107, 'r':         │
│ 504.003, 'b': 444.919, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 446]}]}], 'headings':      │
│ ['3.1 PDF backends'], 'origin': {'mimetype': 'application/pdf', 'binary_hash':                   │
│ 11465328351749295394, 'filename': '2408.09869v5.pdf'}}                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Chunk 6 ─────────────────────────────────────────────╮
│ 3.2 AI models                                                                                    │
│ As part of Docling, we initially release two highly capable AI models to the open-source         │
│ community, which have been developed and published recently by our team. The first model is a    │
│ layout analysis model, an accurate object-detector for page elements [13]. The second model is   │
│ TableFormer [12, 9], a state-of-the-art table structure recognition model. We provide the        │
│ pre-trained weights (hosted on huggingface) and a separate package for the inference code as     │
│ docling-ibm-models . Both                                                                        │
│                                                                                                  │
│ Metadata:                                                                                        │
│ source: https://arxiv.org/pdf/2408.09869                                                         │
│ dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0',          │
│ 'doc_items': [{'self_ref': '#/texts/50', 'parent': {'$ref': '#/body'}, 'children': [],           │
│ 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 3, 'bbox': {'l': 108.0, 't':      │
│ 404.873, 'r': 504.003, 'b': 330.866, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 608]}]}],    │
│ 'headings': ['3.2 AI models'], 'origin': {'mimetype': 'application/pdf', 'binary_hash':          │
│ 11465328351749295394, 'filename': '2408.09869v5.pdf'}}                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Chunk 7 ─────────────────────────────────────────────╮
│ Layout Analysis Model                                                                            │
│ Our layout analysis model is an object-detector which predicts the bounding-boxes and classes of │
│ various elements on the image of a given page. Its architecture is derived from RT-DETR [16] and │
│ re-trained on DocLayNet [13], our popular human-annotated dataset for document-layout analysis,  │
│ among other proprietary datasets. For inference, our implementation relies on the onnxruntime    │
│ [5].                                                                                             │
│ The Docling pipeline feeds page images at 72 dpi resolution, which can be processed on a         │
│                                                                                                  │
│ Metadata:                                                                                        │
│ source: https://arxiv.org/pdf/2408.09869                                                         │
│ dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0',          │
│ 'doc_items': [{'self_ref': '#/texts/52', 'parent': {'$ref': '#/body'}, 'children': [],           │
│ 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 3, 'bbox': {'l': 108.0, 't':      │
│ 293.51, 'r': 504.003, 'b': 252.231, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 388]}]},      │
│ {'self_ref': '#/texts/53', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer':        │
│ 'body', 'label': 'text', 'prov': [{'page_no': 3, 'bbox': {'l': 108.0, 't': 244.394, 'r':         │
│ 504.003, 'b': 192.20600000000002, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 443]}]}],       │
│ 'headings': ['Layout Analysis Model'], 'origin': {'mimetype': 'application/pdf', 'binary_hash':  │
│ 11465328351749295394, 'filename': '2408.09869v5.pdf'}}                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Chunk 8 ─────────────────────────────────────────────╮
│ Table Structure Recognition                                                                      │
│ The TableFormer model [12], first published in 2022 and since refined with a custom structure    │
│ token language [9], is a vision-transformer model for table structure recovery. It can predict   │
│ the logical row and column structure of a given table based on an input image, and determine     │
│ which table cells belong to column headers, row headers or the table body. Compared to earlier   │
│ approaches, TableFormer handles many characteristics of tables, such as partial or no borderlin  │
│                                                                                                  │
│ Metadata:                                                                                        │
│ source: https://arxiv.org/pdf/2408.09869                                                         │
│ dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0',          │
│ 'doc_items': [{'self_ref': '#/texts/55', 'parent': {'$ref': '#/body'}, 'children': [],           │
│ 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 3, 'bbox': {'l': 108.0, 't':      │
│ 154.85000000000002, 'r': 504.003, 'b': 69.93399999999997, 'coord_origin': 'BOTTOMLEFT'},         │
│ 'charspan': [0, 706]}]}, {'self_ref': '#/texts/57', 'parent': {'$ref': '#/body'}, 'children':    │
│ [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 4, 'bbox': {'l': 108.0, 't':  │
│ 716.523, 'r': 504.003, 'b': 664.335, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 459]}]}],    │
│ 'headings': ['Table Structure Recognition'], 'origin': {'mimetype': 'application/pdf',           │
│ 'binary_hash': 11465328351749295394, 'filename': '2408.09869v5.pdf'}}                            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Chunk 9 ─────────────────────────────────────────────╮
│ OCR                                                                                              │
│ Docling provides optional support for OCR, for example to cover scanned PDFs or content in       │
│ bitmaps images embedded on a page. In our initial release, we rely on EasyOCR [1], a popular     │
│ thirdparty OCR library with support for many languages. Docling, by default, feeds a             │
│ high-resolution page image (216 dpi) to the OCR engine, to allow capturing small print detail in │
│ decent quality. While EasyOCR delivers reasonable transcription quality, we observe that it runs │
│ fairly slow on CPU (upwards of 30                                                                │
│                                                                                                  │
│ Metadata:                                                                                        │
│ source: https://arxiv.org/pdf/2408.09869                                                         │
│ dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0',          │
│ 'doc_items': [{'self_ref': '#/texts/59', 'parent': {'$ref': '#/body'}, 'children': [],           │
│ 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 4, 'bbox': {'l': 108.0, 't':      │
│ 631.616, 'r': 504.003, 'b': 568.518, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 515]}]},     │
│ {'self_ref': '#/texts/60', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer':        │
│ 'body', 'label': 'text', 'prov': [{'page_no': 4, 'bbox': {'l': 108.0, 't': 560.682, 'r':         │
│ 504.003, 'b': 541.221, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 139]}]}], 'headings':      │
│ ['OCR'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 11465328351749295394,          │
│ 'filename': '2408.09869v5.pdf'}}                                                                 │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

#### Ingest into vector store

In [29]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [30]:
# %% [markdown]
# # Step 5. Build Vector Store (Milvus with OpenAI Embeddings)

# %%
from langchain_openai import OpenAIEmbeddings
from langchain_milvus import Milvus

embeddings = OpenAIEmbeddings(model=EMBED_MODEL_ID)

vectorstore = Milvus.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="docling_demo",
    connection_args={"uri": MILVUS_URI},
    index_params={"index_type": "FLAT"},
    drop_old=True,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})


#### Define LLM

In [31]:
# %% [markdown]
# # Step 6. Define LLM (OpenAI GPT)

# %%
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=GEN_MODEL_ID, temperature=0)


#### RAG chain

In [32]:
# %% [markdown]
# # Step 7. Create RAG Chain

# %%
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

def clip_text(text, threshold=100):
    return f"{text[:threshold]}..." if len(text) > threshold else text

question_answer_chain = create_stuff_documents_chain(llm, PROMPT)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)


### QnA

In [33]:
# %% [markdown]
# # Step 8. Ask a Question

# %%
resp_dict = rag_chain.invoke({"input": QUESTION})

clipped_answer = clip_text(resp_dict["answer"], threshold=350)
print(f"Question:\n{resp_dict['input']}\n\nAnswer:\n{clipped_answer}")

for i, doc in enumerate(resp_dict["context"]):
    print()
    print(f"Source {i + 1}:")
    print(f"  text: {clip_text(doc.page_content, threshold=350)}")
    for key, val in doc.metadata.items():
        if key != "pk":
            clipped_val = clip_text(val) if isinstance(val, str) else val
            print(f"  {key}: {clipped_val}")


Question:
Which are the main AI models in Docling?

Answer:
The main AI models in Docling are:

1. A layout analysis model, which is an accurate object-detector for page elements.
2. TableFormer, a state-of-the-art table structure recognition model.

Source 1:
  text: 3.2 AI models
As part of Docling, we initially release two highly capable AI models to the open-source community, which have been developed and published recently by our team. The first model is a layout analysis model, an accurate object-detector for page elements [13]. The second model is TableFormer [12, 9], a state-of-the-art table structure re...
  source: https://arxiv.org/pdf/2408.09869
  dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/50', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 3, 'bbox': {'l': 108.0, 't': 404.873, 'r': 504.003, 'b': 330.866, 'coord_origin': 'BOTTOMLEFT'

## API reference

- [LangChain Docling integration GitHub](https://github.com/docling-project/docling-langchain)
- [Docling GitHub](https://github.com/docling-project/docling)
- [Docling docs](https://docling-project.github.io/docling//)